
1. GIẢI THÍCH CÁC CHỈ SỐ TRẠNG THÁI (STATE INDICATORS)
Hệ thống tự động chuyển đổi các số liệu đo lường từ cảm biến và vệ tinh thành các nhãn trạng thái sinh lý như sau:

Chỉ số Mặn (S_Lab):

Salt_Low (dưới 2 PSU): Ngưỡng an toàn cho đa số cây trồng nước ngọt.

Salt_Mod (2 - 4 PSU): Ngưỡng bắt đầu gây ức chế sinh trưởng nhẹ.

Salt_High (trên 4 PSU): Ngưỡng gây độc trực tiếp và làm gián đoạn quá trình hấp thụ nước của rễ.

Độ ẩm đất (M_Lab):

Soil_Dry (dưới 0.25 m3/m3): Đất thiếu nước, cây bắt đầu rơi vào tình trạng hán sinh lý.

Soil_Wet (trên 0.25 m3/m3): Độ ẩm đảm bảo cho các hoạt động trao đổi chất diễn ra bình thường.

Mức độ Stress (P_Lab):

Plant_Safe: Cây khỏe mạnh, chỉ số xanh (NDVI) ổn định. Cây quang hợp tốt.

Plant_Warning: Có dấu hiệu suy giảm diệp lục hoặc thiếu nước tạm thời. Cần can thiệp sớm để tránh thiệt hại.

Plant_DANGER: Cây đạt ngưỡng stress cực hạn, nguy cơ héo rũ và chết cây rất cao nếu không được giải mặn kịp thời.

In [7]:
import pandas as pd
import numpy as np
from prefixspan import PrefixSpan

# Load dữ liệu
df = pd.read_csv('merged_final.csv', parse_dates=['date'])
df = df.sort_values('date')

# 1. Định nghĩa nhãn cho Độ mặn (Salinity)
def label_salinity(x):
    if x < 2: return 'Salt_Low'
    if x < 4: return 'Salt_Mod'
    return 'Salt_High'

# 2. Định nghĩa nhãn cho Độ ẩm đất (Soil Moisture)
def label_moisture(x):
    if x < 0.25: return 'Soil_Dry'
    return 'Soil_Wet'

# 3. Định nghĩa nhãn Stress cây trồng (Target)
def label_stress(x):
    if x < 0.4: return 'Plant_Safe'
    if x < 0.7: return 'Plant_Warning'
    return 'Plant_DANGER'

df['S_Lab'] = df['salinity_psu'].apply(label_salinity)
df['M_Lab'] = df['soil_moisture_surface'].apply(label_moisture)
df['P_Lab'] = df['crop_stress_score'].apply(label_stress)

# Tạo một "Sự kiện tổng hợp" (State) cho mỗi ngày
df['State'] = df['S_Lab'] + "|" + df['M_Lab'] + "|" + df['P_Lab']

Chúng ta sẽ tìm các quy luật diễn ra trong vòng 7 ngày dẫn đến trạng thái Plant_DANGER.

In [8]:
# Gom nhóm dữ liệu theo từng chuỗi 7 ngày (Rolling window)
sequence_list = []
window_size = 7

for i in range(len(df) - window_size):
    seq = df['State'].iloc[i : i + window_size].tolist()
    sequence_list.append(seq)

# Chạy PrefixSpan để tìm các chuỗi phổ biến
ps = PrefixSpan(sequence_list)
# Tìm các pattern xuất hiện ít nhất 5% thời gian và có độ dài ít nhất 3 ngày
min_sup = int(len(sequence_list) * 0.05)
patterns = ps.frequent(min_sup)

# Lọc ra các pattern có kết thúc bằng trạng thái DANGER
danger_patterns = [p for p in patterns if 'Plant_DANGER' in p[1][-1] and len(p[1]) >= 3]
danger_patterns = sorted(danger_patterns, key=lambda x: x[0], reverse=True)

. PHÂN CẤP MỨC ĐỘ CẢNH BÁO (ALERT LEVELS)
MỨC XANH (An toàn): Các chuỗi sự kiện nằm trong ngưỡng chịu đựng. Duy trì chế độ canh tác bình thường.

MỨC VÀNG (Cảnh báo): Xuất hiện các chuỗi có tiền tố dẫn đến rủi ro (Ví dụ: Mặn tăng liên tục trong 3 ngày). Kiến nghị kiểm tra độ mặn tại các cửa cống và chuẩn bị nguồn nước dự trữ.

MỨC ĐỎ (Nguy hiểm): Chuỗi hiện tại khớp trên 80% với các kịch bản gây chết cây trong lịch sử. Kiến nghị đóng cống ngăn mặn khẩn cấp và thực hiện các biện pháp giải stress cho cây.

In [9]:
def get_crop_alert(recent_states, danger_patterns):
    alert_level = "GREEN (An toàn)"
    advice = "Môi trường ổn định, cây trồng phát triển bình thường."
    match_count = 0
    
    for support, pattern in danger_patterns:
        # Kiểm tra xem chuỗi hiện tại có khớp với phần đầu của một pattern nguy hiểm không
        prefix = pattern[:-1]
        if recent_states[-len(prefix):] == prefix:
            match_count += 1
            
    if match_count > 0:
        if 'Salt_High' in recent_states[-1]:
            alert_level = "RED (NGUY HIỂM CAO)"
            advice = "Cảnh báo: Độ mặn vượt ngưỡng kết hợp đất khô. Cần xả mặn hoặc ngưng tưới ngay lập tức để cứu cây!"
        else:
            alert_level = "YELLOW (CẢNH BÁO)"
            advice = "Dấu hiệu Stress đang tăng dần. Kiểm tra hệ thống tưới và độ mặn nước sông."
            
    return alert_level, advice

# --- THỰC THI TRÊN DỮ LIỆU MỚI NHẤT ---
current_context = df['State'].tail(3).tolist() # Lấy 3 ngày gần nhất
level, msg = get_crop_alert(current_context, danger_patterns)

print(f"=== HỆ THỐNG GIÁM SÁT SỨC KHỎE CÂY TRỒNG ===")
print(f"Trạng thái hiện tại: {current_context[-1]}")
print(f"MỨC ĐỘ BÁO ĐỘNG: {level}")
print(f"KHUYẾN NGHỊ: {msg}")

=== HỆ THỐNG GIÁM SÁT SỨC KHỎE CÂY TRỒNG ===
Trạng thái hiện tại: Salt_High|Soil_Wet|Plant_Safe
MỨC ĐỘ BÁO ĐỘNG: GREEN (An toàn)
KHUYẾN NGHỊ: Môi trường ổn định, cây trồng phát triển bình thường.
